In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

# Sample dataset
df = pd.read_csv("freelancers_dataset_full.csv")




In [2]:
# Drop multiple columns by specifying a list of column names
df = df.drop(['Freelancer_ID', 'Experience_Years', 'Project_Rate','Rating','Jobs_Completed'], axis=1)

# Rename a column by providing a dictionary mapping old column name to new column name
df = df.rename(columns={'Description': 'bio','Skills':'skills'})

In [3]:
df.to_csv('modified_freelancer.csv', index=False)


In [4]:
df = pd.read_csv("modified_freelancer.csv")


In [5]:
# Split dataset into training and testing
train_bios, test_bios, train_skills, test_skills = train_test_split(df['bio'], df['skills'], test_size=0.2, random_state=42)

# Tokenization and Padding
max_vocab_size = 5000
max_len = 60  # Limit to 20 words per bio for simplicity

tokenizer = Tokenizer(num_words=max_vocab_size, lower=True, split=' ')
tokenizer.fit_on_texts(train_bios)
train_bios_seq = tokenizer.texts_to_sequences(train_bios)
test_bios_seq = tokenizer.texts_to_sequences(test_bios)

train_bios_seq = pad_sequences(train_bios_seq, maxlen=max_len)
test_bios_seq = pad_sequences(test_bios_seq, maxlen=max_len)

# Similarly tokenize the skills (if needed)
tokenizer_skills = Tokenizer(num_words=max_vocab_size, lower=True, split=',')
tokenizer_skills.fit_on_texts(train_skills)
train_skills_seq = tokenizer_skills.texts_to_sequences(train_skills)
test_skills_seq = tokenizer_skills.texts_to_sequences(test_skills)

train_skills_seq = pad_sequences(train_skills_seq, maxlen=max_len)
test_skills_seq = pad_sequences(test_skills_seq, maxlen=max_len)

In [ ]:
# Autoencoder Model
def build_autoencoder(input_dim, encoding_dim=32):
    model = Sequential()
    
    # Encoder
    model.add(Embedding(input_dim=input_dim, output_dim=64, input_length=max_len))  # Embedding layer
    model.add(LSTM(128, activation='relu', return_sequences=False))  # LSTM for sequence learning
    model.add(RepeatVector(max_len))  # Repeat to match the input shape for the decoder
    
    # Decoder
    model.add(LSTM(128, activation='relu', return_sequences=True))  # LSTM layer for reconstruction
    model.add(TimeDistributed(Dense(input_dim, activation='softmax')))  # Dense layer with softmax activation for each time step
    
    # Compile the model
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Build and train the model
autoencoder = build_autoencoder(input_dim=max_vocab_size)
autoencoder.summary()

# Train the model
autoencoder.fit(train_bios_seq, np.expand_dims(train_bios_seq, -1), epochs=50, batch_size=32, validation_data=(test_bios_seq, np.expand_dims(test_bios_seq, -1)))


C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector (RepeatVector)         │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 55s 192ms/step - accuracy: 0.4110 - loss: 8.8764 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 188ms/step - accuracy: 0.4278 - loss: 9.2228 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 188ms/step - accuracy: 0.4277 - loss: 9.2250 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 188ms/step - accuracy: 0.4277 - loss: 9.2250 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 190ms/step - accuracy: 0.4283 - loss: 9.2149 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 192ms/step - accuracy: 0.4285 - loss: 9.2127 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 188ms/step - accuracy: 0.4276 - loss: 9.2270 - val_accuracy: 0.4288 - val_loss: 9.2074
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 46s 183ms/step - accuracy: 0.4283 - loss: 9